In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import math
from torch.utils.data import Dataset, DataLoader, random_split
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix
from torch.optim import AdamW
from tqdm.auto import tqdm
from torchvision.models import resnet18, ResNet18_Weights
from statsmodels.tsa.stattools import grangercausalitytests
from transformers import AutoTokenizer

# ========================= CONFIG =========================
H5_FILE_PATH = "/home/poorna/data/eeg_raw_image_final.h5"
TRAIN_PCT = 0.8
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

D_MODEL = 256
D_EMBEDDING = 128
TEMPERATURE = 0.07
NUM_CHANNELS = 62
EPOCHS_PHASE1 = 20
MODEL_SAVE_PATH = 'eeg_image_contrastive_best.pt'
IMAGE_CHANNELS = 3
IMAGE_SIZE = 224
# ==========================================================

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

# ---------------- Positional Encoding ----------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ---------------- Multi-Head Attention & Transformer Layer ----------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        if mask is not None: mask = mask.unsqueeze(1)
        bs = q.size(0)
        q, k, v = [l(x).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
                   for l, x in zip(self.linears, (q, k, v))]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        x = torch.matmul(attn, v)
        x = x.transpose(1, 2).contiguous().view(bs, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

# ---------------- Granger Graph (FIXED for PyG Edge Attr) ----------------
def create_granger_causality_matrix(eeg_sample_cpu):
    if torch.is_tensor(eeg_sample_cpu):
        eeg_np = eeg_sample_cpu.cpu().numpy()
    else:
        eeg_np = eeg_sample_cpu
    eeg_np = eeg_np.T  # [C, T] -> [T, C]
    n = eeg_np.shape[1]
    adj = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j: continue
            data = np.column_stack([eeg_np[:, j], eeg_np[:, i]])
            try:
                res = grangercausalitytests(data, maxlag=5, verbose=False)
                if res[5][0]['ssr_ftest'][1] < 0.05:
                    adj[i, j] = 1.0
            except:
                pass
    coo = coo_matrix(adj)
    edge_index, edge_attr = from_scipy_sparse_matrix(coo)
    
    # --- FIX 1: Ensure edge_attr is a simple 1D tensor of weights (1.0)
    if edge_attr is not None:
        # We only care about the existence of the edge, not complex features
        edge_attr = torch.ones(edge_attr.size(0), dtype=torch.float)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float) if edge_attr is not None else None

# ---------------- Dataset ----------------
class EEGImageH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))       # [C, T]
        img = torch.from_numpy(self.h5_file['image_pixels'][idx].astype('float32'))
        if img.shape[-1] == 3:
            img = img.permute(2, 0, 1)
        img = img / 255.0 if img.max() > 1.0 else img
        return eeg, img

def collate_fn(batch):
    eeg, img = zip(*batch)
    return torch.stack(eeg), torch.stack(img)

# ---------------- EEG Encoder (GCN Batching Logic Confirmed) ----------------
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=NUM_CHANNELS, d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = get_clones(TransformerEncoderLayer(d_model, num_heads, d_ff, dropout), num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def _batch_graph(self, edge_index_single, edge_attr_single, batch_size, seq_len):
        """Robustly batches the static graph structure across B*T nodes."""
        device = edge_index_single.device
        n_nodes = self.num_channels
        n_edges = edge_index_single.size(1)
        total_graphs = batch_size * seq_len

        edge_index = edge_index_single.repeat(1, total_graphs)
        
        # NOTE: edge_attr is 1D tensor of weights (size N_edges)
        edge_attr = edge_attr_single.repeat(total_graphs) if edge_attr_single is not None else None

        offset = torch.arange(total_graphs, device=device) * n_nodes
        offset = offset.repeat_interleave(n_edges).unsqueeze(0)
        edge_index = edge_index + offset
        
        return edge_index.contiguous(), edge_attr.contiguous() if edge_attr is not None else None

    def forward(self, x, edge_index, edge_attr):      # x: [B, C, T]
        B, C, T = x.shape
        edge_idx_batched, edge_attr_batched = self._batch_graph(edge_index, edge_attr, B, T)

        x = x.permute(0, 2, 1).reshape(B * T, C)          # [B×T, C]
        x = F.relu(self.gcn1(x, edge_idx_batched, edge_attr_batched))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, edge_idx_batched, edge_attr_batched))
        x = x.view(B, T, self.d_model)

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                    # [B, 1+T, D]
        x = self.pos_enc(x)
        for layer in self.transformer:
            x = layer(x)
        x = self.norm(x)
        return x

# ---------------- Image Encoder ----------------
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=D_EMBEDDING):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        for p in backbone.parameters(): p.requires_grad = False
        self.backbone = backbone
        self.head = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, out_dim))
    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

# ---------------- Dual Encoder ----------------
class DualEncoderContrastive(nn.Module):
    def __init__(self):
        super().__init__()
        self.eeg_enc = SpatioTemporalEEGEncoderTF()
        self.img_enc = ImageEncoder()
        self.proj = nn.Sequential(nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, D_EMBEDDING))
    def forward(self, eeg, img, edge_index, edge_attr):
        eeg_out = self.eeg_enc(eeg, edge_index, edge_attr)   # [B, 1+T, D]
        eeg_emb = self.proj(eeg_out[:, 0])
        img_emb = self.img_enc(img)
        return F.normalize(eeg_emb, dim=-1), F.normalize(img_emb, dim=-1)

def info_nce(z1, z2, temp=TEMPERATURE):
    logits = torch.matmul(z1, z2.T) / temp
    labels = torch.arange(z1.size(0), device=z1.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# ---------------- Setup ----------------
def setup_data_and_graph():
    dataset = EEGImageH5Dataset(H5_FILE_PATH)
    n_train = int(len(dataset) * TRAIN_PCT)
    train_ds, val_ds = random_split(dataset, [n_train, len(dataset)-n_train],
                                    generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    # Compute Granger graph on CPU from the first sample
    eeg_sample, _ = next(iter(train_loader))      # [B, C, T] on CPU
    edge_index, edge_attr = create_granger_causality_matrix(eeg_sample[0]) # only first trial

    edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr,
                                           fill_value=1.0, num_nodes=NUM_CHANNELS)

    edge_index = edge_index.to(device).contiguous()
    edge_attr  = edge_attr.to(device).contiguous() if edge_attr is not None else None

    return train_loader, val_loader, edge_index, edge_attr

# ---------------- Training ----------------
def run_phase1_contrastive():
    print("\n" + "="*60)
    print("STARTING PHASE 1: EEG ↔ Image Contrastive Learning")
    print("="*60)

    train_loader, val_loader, edge_index, edge_attr = setup_data_and_graph()
    model = DualEncoderContrastive().to(device)
    opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    best_val = float('inf')
    for epoch in range(1, EPOCHS_PHASE1 + 1):
        model.train()
        train_loss = 0.0
        for eeg, img in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
            eeg, img = eeg.to(device), img.to(device)
            opt.zero_grad()
            z_eeg, z_img = model(eeg, img, edge_index, edge_attr)
            loss = info_nce(z_eeg, z_img)
            loss.backward()
            opt.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for eeg, img in val_loader:
                eeg, img = eeg.to(device), img.to(device)
                z_eeg, z_img = model(eeg, img, edge_index, edge_attr)
                val_loss += info_nce(z_eeg, z_img).item()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        print(f"Epoch {epoch:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"    → Best model saved ({best_val:.4f})")

    print("Training finished! Best model:", MODEL_SAVE_PATH)

if __name__ == '__main__':
    try:
        run_phase1_contrastive()
    except Exception as e:
        import traceback
        traceback.print_exc()


STARTING PHASE 1: EEG ↔ Image Contrastive Learning


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Epoch 1 Train:   0%|          | 0/700 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,0], thread: [96,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,0], thread: [97,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,0], thread: [98,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,0], thread: [99,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,0], thread: [100,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [33,0,